In [ ]:
import pandas as pd
import os
import sys

In [ ]:
final_cohorts = pd.read_csv('updated_concept_query_list_greater_than_100.csv')
final_cohorts
concept_id = list(final_cohorts['condition_concept_id'])
len(concept_id)

In [ ]:
def get_cohort_persons_table(concept_id):
    """
    Fetches per-patient summary for the specified condition_concept_id,
    applying Cohort Builder UI filters (EHR + genomics, observation window,
    flat-events, standard concepts), and returns a DataFrame with one row per
    patient including:
      - condition_concept_id
      - standard_concept_name, standard_vocabulary
    """
    dataset = os.environ["WORKSPACE_CDR"]
    sql = f"""
    WITH
      ehr_genomics_patients AS (
        SELECT DISTINCT person_id
        FROM `{dataset}.cb_search_person`
        WHERE has_ehr_data = 1
          AND (
               has_whole_genome_variant      = 1
            OR has_lr_whole_genome_variant   = 1
            OR has_array_data                = 1
          )
      ),

      all_occ AS (
        SELECT
          co.person_id,
          co.condition_concept_id,
          co.visit_occurrence_id,
          co.condition_start_datetime,
          co.condition_end_datetime,
          co.condition_type_concept_id
        FROM `{dataset}.condition_occurrence` co
        JOIN ehr_genomics_patients eg
          ON co.person_id = eg.person_id

        -- only events that made it into the CB search table
        JOIN `{dataset}.cb_search_all_events` ev
          ON ev.person_id = co.person_id
         AND ev.concept_id = co.condition_concept_id
         AND DATE(co.condition_start_datetime) = ev.entry_date

        JOIN `{dataset}.concept` c_std
          ON co.condition_concept_id = c_std.concept_id
        WHERE c_std.standard_concept = 'S'
      ),

      spec_occ AS (
        SELECT *
        FROM all_occ
        WHERE condition_concept_id = {concept_id}
      ),

      detail AS (
        SELECT
          person_id,
          condition_concept_id,
          condition_start_datetime AS first_diag_date,
          condition_end_datetime   AS first_end_date,
          condition_type_concept_id,
          visit_occurrence_id,
          ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY condition_start_datetime) AS rn
        FROM spec_occ
      ),

      first_detail AS (
        SELECT
          d.person_id,
          d.condition_concept_id,
          d.first_diag_date,
          d.first_end_date,
          c_std.concept_name        AS standard_concept_name,
          c_std.vocabulary_id       AS standard_vocabulary,
          c_type.concept_name       AS condition_type_concept_name,
          vis_evt.concept_name      AS visit_occurrence_concept_name
        FROM detail d
        JOIN `{dataset}.concept` c_std
          ON d.condition_concept_id = c_std.concept_id
        LEFT JOIN `{dataset}.concept` c_type
          ON d.condition_type_concept_id = c_type.concept_id
        LEFT JOIN `{dataset}.visit_occurrence` v
          ON d.visit_occurrence_id = v.visit_occurrence_id
        LEFT JOIN `{dataset}.concept` vis_evt
          ON v.visit_concept_id = vis_evt.concept_id
        WHERE d.rn = 1
      ),

      spec_metrics AS (
        SELECT
          person_id,
          MIN(condition_start_datetime) AS first_diag_date,
          MAX(condition_start_datetime) AS last_diag_date,
          COUNT(DISTINCT visit_occurrence_id) AS visits_for_concept
        FROM spec_occ
        GROUP BY person_id
      ),

      allv AS (
        SELECT
          person_id,
          COUNT(DISTINCT visit_occurrence_id) AS visits_all_concepts
        FROM all_occ
        GROUP BY person_id
      ),

      conc AS (
        SELECT
          person_id,
          COUNT(DISTINCT condition_concept_id) AS concept_count_ehr
        FROM all_occ
        GROUP BY person_id
      )

    SELECT
      fd.person_id,
      fd.condition_concept_id,
      fd.standard_concept_name,
    FROM first_detail fd
    JOIN spec_metrics sm  ON fd.person_id = sm.person_id
    LEFT JOIN allv av       ON fd.person_id = av.person_id
    LEFT JOIN conc cc       ON fd.person_id = cc.person_id
    """

    df = pd.read_gbq(
        sql,
        project_id=os.environ.get("BIGQUERY_PROJECT"),
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook",
    )
    return df

In [ ]:
#getting data tables with list of person_ID and concept_names 


cohort_IDs = dict()
    
def get_cohort_IDs():
    
    final_cohorts = pd.read_csv('updated_concept_query_list_greater_than_100.csv')
    condition_ids = list(final_cohorts['condition_concept_id'])
    
    for concept_id in condition_ids:
    
        cond = get_cohort_persons_table(concept_id)
        
        cohort_IDs[concept_id] = cond
        

    return cohort_IDs
    
    

In [ ]:
cohort_person_IDs = get_cohort_IDs()

In [ ]:
cohort_person_IDs

In [ ]:

viral_overlap = dict()
    
def viral_overlap_dict(dict_of_cohort_IDs):
    
    
    
    for key, table in cohort_person_IDs.items():
        
        person_IDs = table['person_id']
        
        concept_names_series = table['standard_concept_name']
        concept_names_str = concept_names_series[0]
    
        viral_overlap[concept_names_str]= set(person_IDs)
        
        
    return viral_overlap
    
    

    

In [ ]:
viral_dict = viral_overlap_dict(cohort_person_IDs)
viral_dict

In [ ]:
def viral_overlap_count(viral_overlap):
    
    max_overlap = []
    all_concept_names = viral_overlap.keys() #a list of dictionary keys (concept_names)
    less_than_20 = set()
    
        
    for concept_name in all_concept_names: #concept_name = concept name per row for each item in all_cocnept_names list
        max_concept_name = None
        overlap = 0
        person_ids = viral_overlap[concept_name] #list of person_id asscoiated with each key in viral_overlap dict
        person_ids_count = len(person_ids) #count of person_id asscociated with each key

        
        #initialize in case no overlap is found
        max_concept_count = 0
        overlap_percent = 0
        max_concept_overlap_percent = 0
        average = 0
        
        
        
        for concept_name_to_check in all_concept_names: #all concept names =  all the concept names again for each concept name indivdually in a list
                                                        #concept_name_to_check = each of those concept names individually from the list of concept_names used to check for each original key 

            person_ids_to_check = viral_overlap[concept_name_to_check] #person_ids associated with each concept_name , in the list of concept names to check for each original key
            number_of_overlaps = len(person_ids.intersection(person_ids_to_check)) #the overlap of those person ids, length of list of intersection of two list
            

        
            
            if person_ids == person_ids_to_check: #skipping self comparison (logical error)
                continue
            
            
        
            if number_of_overlaps > overlap:  #number_of_overlaps = if greater than 0, then overlap now = # of overlaps #upates the variable 
               
                overlap = number_of_overlaps #number of overlaps between each concept name where the # of overlaps is the greatest
                max_concept_name = concept_name_to_check #concep_name where the # of overlaps is the greatest 
            
                overlap_percent = int((overlap/person_ids_count)*100)  # (number of max_overlap/ person_id count )*100
                max_concept_count = len(viral_overlap[max_concept_name]) #count of person_id in the max_concept_name
                max_concept_overlap_percent = int((overlap/max_concept_count)*100) # (number of max_overlap/ max_concept person_id count )*100
                average = int((overlap_percent + max_concept_overlap_percent)/2) # (overlap_percent + max_concept_overlap_percent) / 2
      
                
        max_overlap.append([concept_name, person_ids_count, max_concept_name, max_concept_count, overlap, overlap_percent, max_concept_overlap_percent, average])
       
        
        
        
    return(max_overlap)


In [ ]:
final = viral_overlap_count(viral_overlap)
final

In [ ]:
final_df = pd.DataFrame(final)
final_df.head(5)
#final_df.shape

In [ ]:
final_df.columns = ['standard_concept_name', 'count', 'max_concept_name', 'max_concept_count', 'overlap_count', 'overlap_percent', 'max_concept_overlap_percent', 'average']


In [ ]:
final_df.head(20)

In [ ]:
final_df['concept_id'] = concept_id

In [ ]:
final_df.columns

In [ ]:
final_df = final_df[['concept_id', 'standard_concept_name', 'count', 'max_concept_name',
       'max_concept_count', 'overlap_count', 'overlap_percent',
       'max_concept_overlap_percent', 'average',]]
final_df

In [ ]:
final_df.to_csv("viral_cohort_patient_overlap.csv")